# SAE Model Scoping — End-to-End Colab Experiment (Steps 1–7)

This notebook walks through **Steps 1–7** from the README end-to-end on Google Colab,
producing a chemistry-scoped Gemma-2-9B-IT model and evaluating it against two OOD tasks.

**What you will do:**
1. ✅ Fork the repo (prerequisite — done outside this notebook)
2. ✅ Install the package and verify imports
3. 🗂️ Pick the in-domain dataset (chemistry) and OOD datasets (coding, general chat)
4. 🔥 Calculate SAE firing rates on the training set and plot them
5. 📊 Evaluate the SAE-hooked model at varying numbers of retained neurons K
6. 🏋️ Train the LLM layers after the SAE to recover in-domain performance
7. 📈 Evaluate the finetuned scoped model in-domain and OOD

## ⚙️ Requirements

| Requirement | Details |
|---|---|
| **GPU** | A100 40 GB recommended (Colab Pro+). V100 16 GB may work with smaller batch. Gemma-2-9B in bfloat16 needs ~18 GB VRAM. |
| **HuggingFace token** | Accept the [Gemma-2-9B-IT licence](https://huggingface.co/google/gemma-2-9b-it) then store your token in Colab Secrets as `HF_TOKEN`. |
| **WandB** (optional) | Store your key in Colab Secrets as `WANDB_API_KEY`. If absent, training runs in offline mode. |
| **Time** | ~30 min firing rates · ~15 min K-sweep · ~2-4 h training (500 steps) · ~15 min eval |

> **Tip:** `Runtime → Change runtime type → A100 GPU`

## ⬇️ Installation

In [ ]:
# Clone the repo (skip if already cloned)
# ⚠️  Replace the URL below with YOUR fork's URL
import os
REPO_URL = "https://github.com/Valpip123EMY/SAEScopingMiniproject.git"
if not os.path.exists("/content/SAEScopingMiniproject"):
    !git clone {REPO_URL}
%cd /content/SAEScopingMiniproject
!pip install -e . -q

In [ ]:
# ── Secrets (HuggingFace + WandB) ────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    HF_TOKEN      = userdata.get("HF_TOKEN") or ""
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception:
    HF_TOKEN      = ""   # paste your HF token here if not using Colab secrets
    WANDB_API_KEY = ""   # paste your WandB key here if not using Colab secrets

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("⚠️  No HF_TOKEN found — Gemma download will fail unless you are already logged in.")

WANDB_PROJECT = "sae-scoping-colab"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ["WANDB_MODE"]    = "online"
    print("WandB: online mode")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WandB: offline mode (no API key)")

## Step 2 — Verify Installation

In [ ]:
import gc
import json
from contextlib import nullcontext
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from safetensors.torch import load_file, save_file
from sae_lens import SAE
from transformers import AutoTokenizer, Gemma2ForCausalLM, PreTrainedTokenizerBase
from trl import SFTConfig

from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.trainers.sae_enhanced.rank import rank_neurons
from sae_scoping.trainers.sae_enhanced.train import train_sae_enhanced_model
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)")
print(f"Device: {DEVICE}")

## Step 1 — Fork the Repo ✅

This step is done outside Colab: fork
[Valpip123EMY/SAEScopingMiniproject](https://github.com/Valpip123EMY/SAEScopingMiniproject)
on GitHub, then update the clone URL in the Install cell above to point to your fork.

## Step 3 — Pick a Dataset

**In-domain (ID):** `4gate/StemQAMixture` — `chemistry` config.
We format each example as `Question: …\nAnswer: …` for SFT.

**Out-of-domain (OOD):**
- `codeparrot/apps` — Python coding problems (very different domain from chemistry)
- `HuggingFaceH4/ultrachat_200k` — general multi-turn chat (broad coverage)

These two OOD datasets were chosen because they represent distinct capability axes
(coding and general instruction-following) that a chemistry-scoped model should lose.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
IN_DOMAIN       = "chemistry"   # change to "biology" if preferred
EVAL_SIZE       = 200           # hold-out examples per dataset
TRAIN_SIZE      = 2000          # in-domain training examples
N_FIRING        = 200           # examples used to compute firing rates
MAX_LENGTH      = 512           # token context length (GPU memory trade-off)
SAE_LAYER       = 31            # which Gemma-2 layer to hook
SAE_ID          = f"layer_{SAE_LAYER}/width_16k/canonical"
HOOKPOINT       = f"model.layers.{SAE_LAYER}"
SAE_RELEASE     = "gemma-scope-9b-pt-res-canonical"
MODEL_NAME      = "google/gemma-2-9b-it"
CACHE_DIR       = Path("/content/cache") / SAE_ID.replace("/", "--")

print("Loading datasets …")
chemistry_raw  = load_dataset("4gate/StemQAMixture", "chemistry", split="train")
apps_raw       = load_dataset("codeparrot/apps",              split="train")
ultrachat_raw  = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

def _fmt_stemqa(ex):
    return {"text": f"Question: {ex['question']}\nAnswer: {ex['answer']}"}

_apps_parse_errors = 0

def _fmt_apps(ex):
    global _apps_parse_errors
    try:
        sols = json.loads(ex.get("solutions", "[]"))
        sol  = sols[0] if sols else ""
    except (ValueError, TypeError):
        _apps_parse_errors += 1
        sol = ""
    return {"text": f"Question: {ex['question']}\nSolution: {sol}"}

def _fmt_ultrachat(ex):
    return {"text": "\n".join(
        f"{m.get('role','unknown').capitalize()}: {m.get('content','')}"
        for m in ex["messages"]
    )}

chem_fmt      = chemistry_raw.map(_fmt_stemqa,   remove_columns=chemistry_raw.column_names)
apps_fmt      = apps_raw.map(_fmt_apps,           remove_columns=apps_raw.column_names)
ultrachat_fmt = ultrachat_raw.map(_fmt_ultrachat, remove_columns=ultrachat_raw.column_names)

def _split(ds, eval_size, train_size=None):
    s     = ds.train_test_split(test_size=min(eval_size, len(ds) - 1), seed=42)
    train = s["train"].select(range(min(train_size or len(s["train"]), len(s["train"]))))
    return {"train": train, "test": s["test"]}

chem_split      = _split(chem_fmt,      EVAL_SIZE, TRAIN_SIZE)
apps_split      = _split(apps_fmt,      EVAL_SIZE)
ultrachat_split = _split(ultrachat_fmt, EVAL_SIZE)

print(f"Chemistry  — train: {len(chem_split['train']):,}  test: {len(chem_split['test']):,}")
print(f"Apps       — test:  {len(apps_split['test']):,}")
print(f"UltraChat  — test:  {len(ultrachat_split['test']):,}")

if _apps_parse_errors:
    print(f"⚠️  {_apps_parse_errors} apps examples had unparseable solutions (treated as empty).")

In [ ]:
# Preview in-domain and OOD examples
print("=== In-domain (chemistry) ===")
print(chem_split["train"]["text"][0][:400])
print("\n=== OOD: apps (coding) ===")
print(apps_split["test"]["text"][0][:300])
print("\n=== OOD: ultrachat (chat) ===")
print(ultrachat_split["test"]["text"][0][:300])

## Step 4 — Calculate SAE Firing Rates

We pass `N_FIRING` in-domain training examples through the model while hooking the SAE
at `model.layers.31`. For each SAE feature (neuron) we count how often it fires above
threshold T=0, then normalise to get a fraction.

In [ ]:
# ── Load tokenizer and model ───────────────────────────────────────────────────
print("Loading tokeniser …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
assert isinstance(tokenizer, PreTrainedTokenizerBase)

print("Loading model (this may take a few minutes) …")
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
    token=HF_TOKEN or None,
)
model = model.to(DEVICE)
model.eval()
print("Model ready.")

In [ ]:
# ── Load SAE ──────────────────────────────────────────────────────────────────
print(f"Loading SAE: {SAE_ID} …")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae.to(DEVICE)
print(f"SAE: d_in={sae.cfg.d_in}  d_sae={sae.cfg.d_sae}")

In [ ]:
# ── Compute firing rates ───────────────────────────────────────────────────────
# Re-use cached results if available
DIST_PATH = CACHE_DIR / "distribution.safetensors"

if DIST_PATH.exists():
    print(f"Loading cached firing rates from {DIST_PATH}")
    data         = load_file(DIST_PATH)
    distribution = data["distribution"].cpu()
    ranking      = data["ranking"].cpu()
else:
    firing_data = chem_split["train"].select(range(min(N_FIRING, len(chem_split["train"]))))
    print(f"Computing firing rates on {len(firing_data)} examples …")
    with torch.no_grad():
        ranking, distribution = rank_neurons(
            dataset=firing_data,
            sae=sae,
            model=model,
            tokenizer=tokenizer,
            T=0.0,
            hookpoint=HOOKPOINT,
            batch_size=4,
            token_selection="attention_mask",
            return_distribution=True,
        )
    ranking      = ranking.detach().cpu()
    distribution = distribution.detach().cpu()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    save_file({"distribution": distribution, "ranking": ranking}, DIST_PATH)
    print(f"Saved to {DIST_PATH}")

print(f"Max firing rate : {distribution.max().item():.6f}")
print(f"d_sae           : {len(distribution)}")
for thresh in [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]:
    n = (distribution >= thresh).sum().item()
    print(f"  threshold={thresh:.0e} → {n:5d} neurons ({100*n/len(distribution):.1f}%)")

In [ ]:
# ── Plot firing rates ─────────────────────────────────────────────────────────
sorted_dist, _ = distribution.sort(descending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sorted rates
axes[0].plot(sorted_dist.numpy())
axes[0].set_yscale("log")
for thresh, col in [(1e-3, "blue"), (1e-4, "red"), (1e-5, "green")]:
    axes[0].axhline(thresh, color=col, linestyle="--", linewidth=0.8, label=f"t={thresh:.0e}")
axes[0].set_title(f"Sorted SAE firing rates — {IN_DOMAIN}")
axes[0].set_xlabel("Feature rank")
axes[0].set_ylabel("Firing rate fraction")
axes[0].legend()

# Cumulative
cumsum = sorted_dist.cumsum(0).numpy()
axes[1].plot(cumsum)
axes[1].axhline(0.90, color="orange", linestyle="--", linewidth=0.8, label="90 %")
axes[1].axhline(0.99, color="red",    linestyle="--", linewidth=0.8, label="99 %")
axes[1].set_title(f"Cumulative firing rate — {IN_DOMAIN}")
axes[1].set_xlabel("Top-k features")
axes[1].set_ylabel("Fraction of total activity")
axes[1].legend()

plt.tight_layout()
plt.savefig("/content/firing_rates.png", dpi=150)
plt.show()
print("Plot saved to /content/firing_rates.png")

## Step 5 — Evaluate SAE-Hooked Model at Varying K

We sweep over a range of K values (neurons retained) and measure the average
cross-entropy loss on the chemistry test set.  We want to find the **smallest K**
where loss stays close to the no-SAE baseline — that is the most aggressive pruning
we can do while keeping in-domain performance intact.

In [ ]:
# ── Helper: average cross-entropy loss ────────────────────────────────────────
@torch.no_grad()
def compute_avg_loss(
    eval_model,
    eval_texts,
    sae_wrapper=None,
    hookpoint=None,
    max_len=MAX_LENGTH,
    batch_size=2,
    n_eval=50,
):
    ctx_mgr = (
        named_forward_hooks(eval_model, {hookpoint: partial(filter_hook_fn, sae_wrapper)})
        if sae_wrapper is not None
        else nullcontext()
    )
    total_loss, n_batches = 0.0, 0
    with ctx_mgr:
        for i in range(0, min(len(eval_texts), n_eval), batch_size):
            batch = tokenizer(
                eval_texts[i : i + batch_size],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_len,
            ).to(DEVICE)
            out = eval_model(**batch, labels=batch["input_ids"])
            total_loss += out.loss.item()
            n_batches  += 1
    return total_loss / max(n_batches, 1)

In [ ]:
# ── Sweep over K values ───────────────────────────────────────────────────────
eval_texts_chem = chem_split["test"]["text"][:50]
N_EVAL          = 50

# Baseline: no SAE hook
baseline_loss = compute_avg_loss(model, eval_texts_chem, n_eval=N_EVAL)
print(f"Baseline (no SAE): {baseline_loss:.4f}")

# SAE in passthrough mode (full model + SAE enc/dec but all neurons kept)
full_sae    = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=len(distribution), T=0.0)
full_sae    = full_sae.to(DEVICE)
full_loss   = compute_avg_loss(model, eval_texts_chem, SAEWrapper(full_sae), HOOKPOINT, n_eval=N_EVAL)
print(f"Full SAE (K=16k):  {full_loss:.4f}")

# Sweep
K_VALUES = [50, 100, 250, 500, 1000, 2000, 4000, 8000, len(distribution)]
k_results = {}
for K in K_VALUES:
    pruned  = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=K, T=0.0).to(DEVICE)
    wrapper = SAEWrapper(pruned)
    loss    = compute_avg_loss(model, eval_texts_chem, wrapper, HOOKPOINT, n_eval=N_EVAL)
    k_results[K] = loss
    print(f"  K={K:>6,}  loss={loss:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_results.keys()), list(k_results.values()), marker="o", label="SAE-hooked")
ax.axhline(baseline_loss, color="green",  linestyle="--", label=f"No-SAE baseline ({baseline_loss:.3f})")
ax.axhline(baseline_loss * 1.05, color="orange", linestyle=":", label="5% degradation budget")
ax.set_xscale("log")
ax.set_xlabel("K (neurons retained)")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"In-domain loss vs K — {IN_DOMAIN} test set")
ax.legend()
plt.tight_layout()
plt.savefig("/content/k_vs_loss.png", dpi=150)
plt.show()
print("Plot saved to /content/k_vs_loss.png")

In [ ]:
# ── Choose K / threshold ─────────────────────────────────────────────────────
BUDGET = 1.05   # accept up to 5% loss increase vs no-SAE baseline
CHOSEN_K = None
for K in sorted(k_results.keys()):
    if k_results[K] <= BUDGET * baseline_loss:
        CHOSEN_K = K
        break

if CHOSEN_K is None:
    CHOSEN_K = max(k_results.keys())
    print(f"⚠️  All K values exceed budget — defaulting to K={CHOSEN_K}.")
else:
    print(f"✅ Chosen K={CHOSEN_K}  (loss={k_results[CHOSEN_K]:.4f}  budget={BUDGET*baseline_loss:.4f})")

# Compute corresponding threshold
CHOSEN_THRESHOLD = distribution[ranking[CHOSEN_K - 1]].item()
print(f"Threshold at K={CHOSEN_K}: {CHOSEN_THRESHOLD:.2e}")

## Step 6 — Train the LLM with Pruned SAE

We freeze all model layers **up to and including** `model.layers.31` (the SAE layer)
and SFT-train only the layers **after** the SAE on in-domain chemistry data.
This recovers any in-domain performance degraded by the pruning, without giving the
model a chance to learn OOD features through the SAE bottleneck.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MAX_STEPS  = 500    # increase to 1000+ for a full experiment
BATCH_SIZE = 2
ACCUM      = 4      # effective batch = BATCH_SIZE * ACCUM = 8
LR         = 2e-5
OUTPUT_DIR = "/content/checkpoint_scoped"
RUN_NAME   = f"scoped-{IN_DOMAIN}-K{CHOSEN_K}"

# Build pruned SAE
pruned_sae = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=CHOSEN_K, T=0.0).to(DEVICE)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM,
    max_steps=MAX_STEPS,
    num_train_epochs=1,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.1,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=MAX_STEPS,
    bf16=True,
    save_total_limit=2,
    report_to="wandb" if os.environ.get("WANDB_MODE") == "online" else "none",
    max_length=MAX_LENGTH,
    gradient_checkpointing=False,
    run_name=RUN_NAME,
)

os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
os.environ["WANDB_RUN_NAME"] = RUN_NAME
print(f"Training for {MAX_STEPS} steps  (K={CHOSEN_K}, lr={LR}, batch={BATCH_SIZE}×{ACCUM})")

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
scoped_model = train_sae_enhanced_model(
    train_dataset=chem_split["train"],
    eval_dataset={
        "chemistry": chem_split["test"],
        "apps":      apps_split["test"],
        "ultrachat": ultrachat_split["test"],
    },
    sae=pruned_sae,
    model=model,                    # modified in-place; also returned below
    tokenizer=tokenizer,
    T=0.0,
    hookpoint=HOOKPOINT,
    save_output=True,
    return_trained_model=True,      # returns the model after training
    trainer_config=sft_config,
    wandb_project_name=WANDB_PROJECT,
    wandb_run_name=RUN_NAME,
)
print("Training complete.  Checkpoint saved to", OUTPUT_DIR)

## Step 7 — Evaluate Finetuned Model In-Domain and OOD

We compare three conditions:

| Model | SAE | Expected in-domain | Expected OOD |
|---|---|---|---|
| Original Gemma-2-9B-IT | None | low loss (capable) | low loss (capable) |
| Original + pruned SAE | K kept | ~same (SAE barely changes it) | ~same |
| **Scoped (finetuned) + pruned SAE** | K kept | **lower loss (better)** | **higher loss (degraded ✅)** |

A successful scoping experiment shows:
- in-domain loss ≤ original baseline (the model didn't get worse at chemistry)
- OOD loss > original baseline (the model got worse at coding / general chat)

In [ ]:
# ── Evaluation helper (reuses compute_avg_loss from Step 5) ─────────────────
pruned_sae_eval = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=CHOSEN_K, T=0.0).to(DEVICE)
sae_wrapper_eval = SAEWrapper(pruned_sae_eval)

EVAL_SETS = {
    "chemistry  (ID)":    chem_split["test"]["text"][:50],
    "apps        (OOD)":  apps_split["test"]["text"][:50],
    "ultrachat   (OOD)":  ultrachat_split["test"]["text"][:50],
}

rows = []
print(f"{'Dataset':<25} {'Original':>12} {'Orig+SAE':>12} {'Scoped+SAE':>12} {'Δ OOD?':>10}")
print("─" * 75)
for name, texts in EVAL_SETS.items():
    orig_no_sae = compute_avg_loss(model,        texts, None,            None,     n_eval=50)
    orig_sae    = compute_avg_loss(model,        texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    scoped_sae  = compute_avg_loss(scoped_model, texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    delta       = "↑ worse (✅)" if scoped_sae > orig_no_sae * 1.01 else "→ similar"
    if "ID" in name:
        delta   = "↓ better ✅" if scoped_sae < orig_no_sae * 1.05 else "→ similar"
    print(f"{name:<25} {orig_no_sae:>12.4f} {orig_sae:>12.4f} {scoped_sae:>12.4f} {delta:>10}")
    rows.append({"dataset": name, "original": orig_no_sae, "orig_sae": orig_sae, "scoped_sae": scoped_sae})

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
names  = [r["dataset"] for r in rows]
orig   = [r["original"]   for r in rows]
o_sae  = [r["orig_sae"]   for r in rows]
sc_sae = [r["scoped_sae"] for r in rows]

x     = range(len(names))
w     = 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w     for i in x], orig,   w, label="Original (no SAE)",   color="steelblue", hatch="")
ax.bar([i         for i in x], o_sae,  w, label="Original + SAE",      color="orange",    hatch="//")
ax.bar([i + w     for i in x], sc_sae, w, label="Scoped + SAE",        color="green",     hatch="xx")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=12, ha="right")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"Step 7 — Scoped model evaluation  (K={CHOSEN_K})")
ax.legend()
plt.tight_layout()
plt.savefig("/content/step7_results.png", dpi=150)
plt.show()
print("Plot saved to /content/step7_results.png")

## Summary & Next Steps

### What to look for
- **In-domain (chemistry):** scoped model should have loss ≤ original model.
  If not, try increasing `MAX_STEPS` or lowering `LR`.
- **OOD (apps / ultrachat):** scoped model should have loss > original model.
  A larger gap means more successful scoping.

### Key hyperparameters to vary
| Parameter | Effect |
|---|---|
| `CHOSEN_K` | Fewer neurons → harder constraint → more OOD degradation but risk in-domain drop |
| `MAX_STEPS` | More steps → better recovery in-domain |
| `LR` | Higher LR → faster but noisier recovery |
| `SAE_LAYER` | Try layer 9, 20, or 31; later layers tend to be more domain-specific |

### Next steps (Steps 8–11)
- **Step 8:** Implement a baseline (e.g. magnitude pruning or unlearning).
- **Step 9:** Argue why your baseline is SoTA.
- **Step 10:** Write up results in Overleaf (≤2 pages + appendix).
- **Step 11 (Bonus):** Extend to Gemma-3 with GemmaScope-2.